# U.S. state insurance statutes — folder layout + how to fill them

**There is no single free API** that serves every state’s insurance code the way California’s **leginfo** does for INS. Sites differ (Angular, West, Lexis, PDF-only). This project **indexes whatever you put under `data/`**, so the practical approach is:

1. **One folder per state:** `data/<state>/ins_codes/*.md` using the same style as California: lowercase full names (`california`, `texas`, `new_york`, …).
2. **Add markdown** (or `.txt`) statute excerpts the same way you did for California (`ca_insurance_code_official.ipynb`).
3. **Re-run ingest** (`python3 -m app.ingest` or **Re-index** in the UI).

**Finding official text:** start from your state legislature’s **session laws / codified statutes** site (the [NCSL legislature list](https://www.ncsl.org/lsn/state-legislature-websites.aspx) links to each capitol). Locate the **Insurance** title or chapter, then export or copy into `.md` files.

**Retrieval:** chunks under any path containing **`ins_codes`** get a small hybrid-score boost (configurable via `RETRIEVAL_PATH_BONUS_SUBSTRINGS` in `.env`).

**Automated fetch (preferred):** from the project root run  
`python scripts/fetch_us_insurance_statutes.py --states all`  
That downloads **California** (leginfo INS), **Virginia** (Title 38.2 split), **Oregon** (ORS chapters 740–750), and writes **`_OFFICIAL_SOURCE.md`** stubs with legislature links for every other state. Then run **`python -m app.ingest`**.

Faster variant (only the three automated fetchers, no stubs):  
`python scripts/fetch_us_insurance_statutes.py --implementations-only`

Run the next cell once to create empty `ins_codes` folders for **all 50 states + DC** (if you have not already).

In [1]:
from __future__ import annotations

from pathlib import Path

STATES: list[tuple[str, str]] = [
    ("alabama", "Alabama"), ("alaska", "Alaska"), ("arizona", "Arizona"), ("arkansas", "Arkansas"),
    ("california", "California"), ("colorado", "Colorado"), ("connecticut", "Connecticut"), ("delaware", "Delaware"),
    ("district_of_columbia", "District of Columbia"), ("florida", "Florida"), ("georgia", "Georgia"), ("hawaii", "Hawaii"),
    ("idaho", "Idaho"), ("illinois", "Illinois"), ("indiana", "Indiana"), ("iowa", "Iowa"),
    ("kansas", "Kansas"), ("kentucky", "Kentucky"), ("louisiana", "Louisiana"), ("maine", "Maine"),
    ("maryland", "Maryland"), ("massachusetts", "Massachusetts"), ("michigan", "Michigan"), ("minnesota", "Minnesota"),
    ("mississippi", "Mississippi"), ("missouri", "Missouri"), ("montana", "Montana"), ("nebraska", "Nebraska"),
    ("nevada", "Nevada"), ("new_hampshire", "New Hampshire"), ("new_jersey", "New Jersey"), ("new_mexico", "New Mexico"),
    ("new_york", "New York"), ("north_carolina", "North Carolina"), ("north_dakota", "North Dakota"), ("ohio", "Ohio"),
    ("oklahoma", "Oklahoma"), ("oregon", "Oregon"), ("pennsylvania", "Pennsylvania"), ("rhode_island", "Rhode Island"),
    ("south_carolina", "South Carolina"), ("south_dakota", "South Dakota"), ("tennessee", "Tennessee"), ("texas", "Texas"),
    ("utah", "Utah"), ("vermont", "Vermont"), ("virginia", "Virginia"), ("washington", "Washington"),
    ("west_virginia", "West Virginia"), ("wisconsin", "Wisconsin"), ("wyoming", "Wyoming"),
]

root = Path("data")
created = 0
for folder, _name in STATES:
    d = root / folder / "ins_codes"
    if not d.is_dir():
        d.mkdir(parents=True, exist_ok=True)
        created += 1
    marker = d / "_add_statute_markdown_here.txt"
    if not marker.exists():
        marker.write_text(
            "Add .md or .txt excerpts from this state's official insurance code.\n"
            "Then run:  python3 -m app.ingest\n",
            encoding="utf-8",
        )

print(f"Ensured {len(STATES)} state ins_codes folders under data/<state>/ins_codes/ (new dirs: {created}).")

Ensured 51 state ins_codes folders under data/<state>/ins_codes/ (new dirs: 0).


## California (already working)

Keep using **`ca_insurance_code_official.ipynb`** → writes into **`data/california/ins_codes/`** (your existing layout).

## Other states

- **Texas:** codified **Insurance Code** is often browsed at [Texas Constitution and Statutes — Insurance Code (IN)](https://statutes.capitol.texas.gov/?link=IN) (site is **JavaScript-heavy**; you may need Playwright or manual copy into `data/texas/ins_codes/`).
- **Florida / New York / …:** use that state’s **official** statutes portal from the NCSL link above, then mirror the same **`.md` per section** pattern you like.

If you later want **automated** downloads for TX/NY/…, add **one notebook per state** (or one module with a plugin dict) once you’ve inspected that site’s HTML or API — we cannot safely guess all 50 parsers in one shot.